# Test: coronary_instance_eomt_small_512_dinov2_skelrecall / augs
Evaluate the best checkpoint on the single_dataset test set.

In [ ]:
import sys, os
os.chdir("/home/dsa/new_seg_final/eomt")

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
from torchvision import tv_tensors
from torchmetrics.detection import MeanAveragePrecision

print("Imports OK, device:", "cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# --- Config ---
CKPT_PATH = "runs/coronary_instance_eomt_small_512_dinov2_skelrecall/augs/checkpoints/best.ckpt"
DATA_ROOT = Path("/home/dsa/new_seg_final/single_dataset")
IMG_DIR = DATA_ROOT / "test" / "images"
LABEL_DIR = DATA_ROOT / "test" / "labels"
IMG_SIZE = (512, 512)
NUM_CLASSES = 9
EVAL_TOP_K = 100
SCORE_THRESH = 0.5

CLASS_NAMES = {
    0: "lad", 1: "lm", 2: "lcx", 3: "lad_b", 4: "lcx_b",
    5: "inter", 6: "rca", 7: "pda", 8: "pborca",
}

CLASS_COLORS = [
    (1.0, 0.0, 0.0),     # lad - red
    (0.0, 1.0, 0.0),     # lm - green
    (0.0, 0.0, 1.0),     # lcx - blue
    (1.0, 1.0, 0.0),     # lad_b - yellow
    (1.0, 0.0, 1.0),     # lcx_b - magenta
    (0.0, 1.0, 1.0),     # inter - cyan
    (1.0, 0.5, 0.0),     # rca - orange
    (0.5, 0.0, 1.0),     # pda - purple
    (0.0, 0.5, 0.0),     # pborca - dark green
]
print("Config OK")

In [ ]:
# --- Load model ---
from training.mask_classification_instance import MaskClassificationInstance
from models.eomt import EoMT
from models.vit import ViT

ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)

encoder = ViT(
    backbone_name="vit_small_patch14_dinov2.lvd142m",
    img_size=IMG_SIZE,
    patch_size=14,
)

network = EoMT(
    encoder=encoder,
    num_classes=NUM_CLASSES,
    num_q=100,
    num_blocks=3,
)

model = MaskClassificationInstance(
    network=network,
    img_size=IMG_SIZE,
    num_classes=NUM_CLASSES,
    attn_mask_annealing_enabled=True,
    attn_mask_annealing_start_steps=[2960, 7400, 11840],
    attn_mask_annealing_end_steps=[7400, 11840, 14800],
)

state_dict = ckpt["state_dict"]
state_dict = {k.replace("._orig_mod.", "."): v for k, v in state_dict.items()}
model.load_state_dict(state_dict, strict=False)
model.eval()
model.cuda()
print("Model loaded!")

In [ ]:
# --- Helper functions ---

def load_gt(label_path, h, w):
    masks, labels = [], []
    if not label_path.exists() or label_path.stat().st_size == 0:
        return masks, labels
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 7:
                continue
            class_id = int(parts[0])
            coords = list(map(float, parts[1:]))
            polygon = []
            for i in range(0, len(coords) - 1, 2):
                polygon.append((coords[i] * w, coords[i + 1] * h))
            if len(polygon) < 3:
                continue
            mask_img = Image.new("L", (w, h), 0)
            ImageDraw.Draw(mask_img).polygon(polygon, fill=1)
            mask = np.array(mask_img, dtype=bool)
            if not mask.any():
                continue
            masks.append(mask)
            labels.append(class_id)
    return masks, labels


def render_masks_on_image(img_np, masks, labels, scores=None, alpha=0.5):
    overlay = img_np.copy().astype(np.float32)
    for mask, label in zip(masks, labels):
        color = np.array(CLASS_COLORS[label]) * 255.0
        for c in range(3):
            overlay[:, :, c] = np.where(
                mask, overlay[:, :, c] * (1 - alpha) + color[c] * alpha, overlay[:, :, c]
            )
    overlay = np.clip(overlay, 0, 255).astype(np.uint8)

    # Draw confidence scores as text annotations
    if scores is not None:
        pil_overlay = Image.fromarray(overlay)
        draw = ImageDraw.Draw(pil_overlay)
        try:
            font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 18)
        except OSError:
            font = ImageFont.load_default()
        outline_r = 2
        for mask, label, score in zip(masks, labels, scores):
            ys, xs = np.where(mask)
            if len(ys) == 0:
                continue
            cy, cx = int(ys.mean()), int(xs.mean())
            text = f"{CLASS_NAMES[label]} {score:.2f}"
            # Thick black outline
            for dx in range(-outline_r, outline_r + 1):
                for dy in range(-outline_r, outline_r + 1):
                    if dx == 0 and dy == 0:
                        continue
                    draw.text((cx + dx, cy + dy), text, fill=(0, 0, 0), font=font)
            r, g, b = [int(c * 255) for c in CLASS_COLORS[label]]
            draw.text((cx, cy), text, fill=(255, 255, 255), font=font)
        overlay = np.array(pil_overlay)

    return overlay


def scale_img_size(size):
    factor = min(IMG_SIZE[0] / size[0], IMG_SIZE[1] / size[1])
    return [round(s * factor) for s in size]


@torch.no_grad()
def predict(img_tensor):
    device = next(model.parameters()).device
    img = img_tensor.to(device)
    orig_h, orig_w = img.shape[-2], img.shape[-1]
    new_h, new_w = scale_img_size((orig_h, orig_w))

    pil_img = Image.fromarray(img.permute(1, 2, 0).cpu().numpy())
    pil_img = pil_img.resize((new_w, new_h), Image.BILINEAR)
    resized = torch.from_numpy(np.array(pil_img)).permute(2, 0, 1).to(device)

    pad_h = max(0, IMG_SIZE[0] - resized.shape[-2])
    pad_w = max(0, IMG_SIZE[1] - resized.shape[-1])
    padded = F.pad(resized, [0, pad_w, 0, pad_h])

    with torch.cuda.amp.autocast():
        mask_logits_all, class_logits_all = model(padded.unsqueeze(0))

    mask_logits = mask_logits_all[-1]
    class_logits = class_logits_all[-1]

    mask_logits = F.interpolate(mask_logits, IMG_SIZE, mode="bilinear")
    mask_logits = mask_logits[:, :, :new_h, :new_w]
    mask_logits = F.interpolate(mask_logits, (orig_h, orig_w), mode="bilinear")

    scores = class_logits[0].softmax(dim=-1)[:, :-1]
    all_labels = torch.arange(NUM_CLASSES, device=device).unsqueeze(0).repeat(scores.shape[0], 1).flatten()
    topk_scores, topk_idx = scores.flatten().topk(min(EVAL_TOP_K, scores.numel()), sorted=True)
    pred_labels = all_labels[topk_idx]
    query_idx = topk_idx // NUM_CLASSES
    ml = mask_logits[0][query_idx]

    masks_bool = ml > 0
    mask_scores = (ml.sigmoid().flatten(1) * masks_bool.flatten(1)).sum(1) / (masks_bool.flatten(1).sum(1) + 1e-6)
    final_scores = topk_scores * mask_scores

    keep = final_scores > SCORE_THRESH
    return masks_bool[keep], pred_labels[keep], final_scores[keep], ml[keep].sigmoid()

print("Helpers OK")

In [ ]:
# --- Evaluate mAP on the full test set ---
metric = MeanAveragePrecision(iou_type="segm")

samples = sorted(IMG_DIR.glob("*.png"))
print(f"Evaluating {len(samples)} test images...")

for i, img_path in enumerate(samples):
    label_path = LABEL_DIR / f"{img_path.stem}.txt"
    pil_img = Image.open(img_path).convert("RGB")
    img_np = np.array(pil_img)
    h, w = img_np.shape[:2]
    img_tensor = tv_tensors.Image(pil_img)

    pred_masks, pred_labels, pred_scores, _ = predict(img_tensor)
    gt_masks, gt_labels = load_gt(label_path, h, w)

    # Build prediction dict
    if pred_masks.numel() > 0:
        preds = [dict(
            masks=pred_masks.cpu(),
            labels=pred_labels.cpu(),
            scores=pred_scores.cpu(),
        )]
    else:
        preds = [dict(
            masks=torch.zeros(0, h, w, dtype=torch.bool),
            labels=torch.zeros(0, dtype=torch.long),
            scores=torch.zeros(0),
        )]

    # Build target dict
    if gt_masks:
        targets = [dict(
            masks=torch.tensor(np.stack(gt_masks), dtype=torch.bool),
            labels=torch.tensor(gt_labels, dtype=torch.long),
        )]
    else:
        targets = [dict(
            masks=torch.zeros(0, h, w, dtype=torch.bool),
            labels=torch.zeros(0, dtype=torch.long),
        )]

    metric.update(preds, targets)
    if (i + 1) % 10 == 0:
        print(f"  [{i+1}/{len(samples)}]")

results = metric.compute()
print("\n=== Test Results (skelrecall/augs best.ckpt) ===")
print(f"  mAP        : {results['map']:.4f}")
print(f"  mAP@50     : {results['map_50']:.4f}")
print(f"  mAP@75     : {results['map_75']:.4f}")
print(f"  mAP_small  : {results['map_small']:.4f}")
print(f"  mAP_medium : {results['map_medium']:.4f}")
print(f"  mAP_large  : {results['map_large']:.4f}")

# Per-class AP
if 'map_per_class' in results and results['map_per_class'].numel() > 1:
    print("\nPer-class AP:")
    for cls_id in range(NUM_CLASSES):
        ap = results['map_per_class'][cls_id].item()
        print(f"  {CLASS_NAMES[cls_id]:>8s}: {ap:.4f}")

In [ ]:
# --- Visualize first 20 test samples ---
NUM_VIS = min(20, len(samples))
vis_samples = [(p, LABEL_DIR / f"{p.stem}.txt") for p in samples[:NUM_VIS]]

fig, axes = plt.subplots(NUM_VIS, 4, figsize=(24, 5 * NUM_VIS))
if NUM_VIS == 1:
    axes = axes[np.newaxis, :]

axes[0, 0].set_title("Original Image", fontsize=16, fontweight="bold")
axes[0, 1].set_title("Ground Truth", fontsize=16, fontweight="bold")
axes[0, 2].set_title("Prediction", fontsize=16, fontweight="bold")
axes[0, 3].set_title("Per-pixel Confidence", fontsize=16, fontweight="bold")

for idx, (img_path, label_path) in enumerate(vis_samples):
    print(f"\r  [{idx+1}/{NUM_VIS}] {img_path.name}", end="")
    pil_img = Image.open(img_path).convert("RGB")
    img_np = np.array(pil_img)
    h, w = img_np.shape[:2]
    img_tensor = tv_tensors.Image(pil_img)

    gt_masks, gt_labels = load_gt(label_path, h, w)
    pred_masks, pred_labels, pred_scores, pred_sigmoid = predict(img_tensor)

    gt_overlay = render_masks_on_image(img_np, gt_masks, gt_labels)
    pred_overlay = render_masks_on_image(
        img_np,
        list(pred_masks.cpu().numpy()),
        list(pred_labels.cpu().numpy()),
        scores=list(pred_scores.cpu().numpy()),
    )

    # Build per-pixel confidence heatmap: max sigmoid across all kept queries
    if pred_sigmoid.numel() > 0:
        confidence_map = pred_sigmoid.cpu().numpy().max(axis=0)
    else:
        confidence_map = np.zeros((h, w), dtype=np.float32)

    axes[idx, 0].imshow(img_np)
    axes[idx, 0].set_ylabel(img_path.stem, fontsize=10, rotation=0, labelpad=60, va="center")
    for ax in axes[idx]:
        ax.set_xticks([])
        ax.set_yticks([])
    axes[idx, 1].imshow(gt_overlay)
    axes[idx, 2].imshow(pred_overlay)
    axes[idx, 3].imshow(img_np, alpha=0.3)
    im = axes[idx, 3].imshow(confidence_map, cmap="hot", vmin=0, vmax=1, alpha=0.7)

plt.colorbar(im, ax=axes[:, 3].tolist(), shrink=0.6, label="Sigmoid confidence")

legend_patches = [mpatches.Patch(color=CLASS_COLORS[i], label=CLASS_NAMES[i]) for i in range(NUM_CLASSES)]
fig.legend(handles=legend_patches, loc="upper center", ncol=NUM_CLASSES, fontsize=12,
           bbox_to_anchor=(0.5, 1.0), frameon=True)

plt.tight_layout(rect=[0, 0, 1, 0.98])
print("\nDone!")
plt.show()